# Week 1 — Hybrid Search: BM25 + Dense Retrieval with RRF Fusion

**Learning objectives**
- Understand why dense-only retrieval fails on exact-match queries
- Implement BM25 from first principles and with `rank-bm25`
- Combine BM25 and dense retrieval using Reciprocal Rank Fusion (RRF)
- Benchmark all three approaches on our golden pension dataset

---

## 1. Why Dense-Only Retrieval Fails

Dense retrieval encodes both query and document into a shared embedding space and retrieves by cosine similarity. This works remarkably well for **semantic** queries. However, it systematically fails when the user asks for something **exact** — a specific article number, a proper noun, or a precise numeric threshold.

**Example failure case** — query: `"Article 14(2) recovery period"`

The dense model embeds this as a general concept about *recovery plans*. It may surface semantically related passages that never contain the literal string `"Article 14(2)"`. BM25, on the other hand, directly rewards term overlap.

In [ ]:
import sys
sys.path.insert(0, "..")

# Sample pension regulation corpus — same source as golden_dataset.json
CORPUS = [
    "Article 12(3): A pension fund shall not distribute any surplus to members or "
    "sponsoring employers unless the policy funding ratio exceeds 110%.",

    "Article 14(1): When the policy funding ratio falls below 104.3%, the pension fund "
    "must notify the supervisory authority within one month and submit a recovery plan "
    "within three months.",

    "Article 14(2): The recovery plan must demonstrate restoration of the funding ratio "
    "to the required level within a period not exceeding ten years, using realistic "
    "actuarial assumptions.",

    "Article 19(1): Pension funds shall invest in accordance with the prudent person "
    "principle. Assets must be invested in the best long-term interests of members.",

    "Article 23(1): Every pension fund with assets exceeding EUR 500 million shall "
    "conduct an Own Risk and Solvency Assessment (ORSA) at least every three years.",

    "Article 31(1): Pension funds shall integrate environmental, social and governance "
    "(ESG) factors into their investment decision-making process.",

    "Article 38(1): Each pension fund must issue an annual Pension Benefit Statement "
    "to every active member within three months of the financial year end.",

    "The prudent person principle requires security, quality, liquidity and "
    "profitability of the portfolio as a whole.",

    "Recovery plans submitted under Article 14 must be approved by the supervisory "
    "authority before implementation.",

    "Indexation of pension rights may be granted when the coverage ratio exceeds "
    "the threshold defined in Article 12(3).",
]

EXACT_QUERY = "Article 14(2) recovery period ten years"
print(f"Query: {EXACT_QUERY!r}")
print(f"Expected top result: index 2 — '{CORPUS[2][:80]}...'")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Dense retrieval — BAAI/bge-small-en-v1.5
dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

BGE_PREFIX = "Represent this sentence for searching relevant passages: "
doc_embeddings = dense_model.encode(CORPUS, normalize_embeddings=True)
query_embedding = dense_model.encode(
    [BGE_PREFIX + EXACT_QUERY], normalize_embeddings=True
)[0]

dense_scores = doc_embeddings @ query_embedding
dense_ranked = np.argsort(dense_scores)[::-1]

print("=== Dense Retrieval Results ===")
for rank, idx in enumerate(dense_ranked[:5]):
    marker = "<-- CORRECT" if idx == 2 else ""
    print(f"  Rank {rank+1} (score={dense_scores[idx]:.4f})  [{idx}] {CORPUS[idx][:70]}... {marker}")

In [ ]:
from rank_bm25 import BM25Okapi

# BM25 retrieval — exact term matching
tokenised_corpus = [doc.lower().split() for doc in CORPUS]
bm25 = BM25Okapi(tokenised_corpus)

bm25_scores = bm25.get_scores(EXACT_QUERY.lower().split())
bm25_ranked = np.argsort(bm25_scores)[::-1]

print("=== BM25 Retrieval Results ===")
for rank, idx in enumerate(bm25_ranked[:5]):
    marker = "<-- CORRECT" if idx == 2 else ""
    print(f"  Rank {rank+1} (score={bm25_scores[idx]:.4f})  [{idx}] {CORPUS[idx][:70]}... {marker}")

print("\n=> BM25 correctly surfaces Article 14(2) at rank 1.")
print("   Dense retrieval may bury it because the embedding captures 'recovery concept'")
print("   but doesn't penalise missing the literal string 'Article 14(2)'.")

## 2. BM25 From First Principles

BM25 (Best Match 25 / Okapi BM25) is a bag-of-words ranking function from the 1994 TREC competition.

### Term Frequency (TF)
Raw TF grows unboundedly — the 100th occurrence of "pension" shouldn't count as much as the 1st. BM25 saturates TF:

$$\text{TF}_{\text{sat}}(t, d) = \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- $f(t,d)$: raw frequency of term $t$ in document $d$
- $k_1 \approx 1.5$: TF saturation; lower → faster saturation
- $b \approx 0.75$: length normalisation; $b=0$ → no normalisation
- $|d|$: document length; $\text{avgdl}$: average document length

### Inverse Document Frequency (IDF)
Terms that appear in every document carry little information:

$$\text{IDF}(t) = \ln\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)$$

- $N$: total number of documents
- $n(t)$: number of documents containing $t$

### BM25 Score
$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \text{TF}_{\text{sat}}(t, d)$$

In [ ]:
import math
from collections import Counter

def bm25_score_manual(query_terms, corpus, k1=1.5, b=0.75):
    """Minimal BM25 implementation to illustrate the formula."""
    N = len(corpus)
    tokenised = [doc.lower().split() for doc in corpus]
    avgdl = sum(len(d) for d in tokenised) / N

    def idf(term):
        n_t = sum(1 for d in tokenised if term in d)
        return math.log((N - n_t + 0.5) / (n_t + 0.5) + 1)

    scores = []
    for doc_tokens in tokenised:
        tf_counter = Counter(doc_tokens)
        dl = len(doc_tokens)
        score = 0.0
        for term in query_terms:
            f = tf_counter.get(term, 0)
            tf_sat = (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))
            score += idf(term) * tf_sat
        scores.append(score)
    return scores

query_terms = EXACT_QUERY.lower().split()
manual_scores = bm25_score_manual(query_terms, CORPUS)
manual_ranked = np.argsort(manual_scores)[::-1]

print("=== Manual BM25 Implementation ===")
for rank, idx in enumerate(manual_ranked[:3]):
    print(f"  Rank {rank+1}: score={manual_scores[idx]:.4f}  [{idx}] {CORPUS[idx][:70]}...")

# Verify our manual result matches rank_bm25
print("\nCorrelation with rank_bm25:", np.corrcoef(manual_scores, bm25_scores)[0, 1].round(4))

## 3. Implementing BM25 with `rank-bm25`

The `rank-bm25` library provides an efficient, vectorised implementation. We'll index the full pension regulation corpus.

In [ ]:
import json
from pathlib import Path
from rank_bm25 import BM25Okapi

# Load the golden dataset to get the regulation corpus
golden_path = Path("../evaluation/golden_dataset.json")
with open(golden_path) as f:
    golden_dataset = json.load(f)

# Extract all unique context passages as our "corpus"
pension_corpus = []
seen = set()
for item in golden_dataset:
    for ctx in item["contexts"]:
        if ctx not in seen:
            pension_corpus.append(ctx)
            seen.add(ctx)

print(f"Corpus size: {len(pension_corpus)} unique passages")
print(f"Sample passage: {pension_corpus[0][:120]}...")

In [ ]:
# Tokenise and build BM25 index
tokenised_pension = [doc.lower().split() for doc in pension_corpus]
bm25_pension = BM25Okapi(tokenised_pension)

# Test with the Article 14(2) query
test_query = "Article 14(2) maximum recovery period"
scores = bm25_pension.get_scores(test_query.lower().split())
ranked_idx = np.argsort(scores)[::-1]

print(f"Query: {test_query!r}\n")
print("Top-3 BM25 results:")
for rank, idx in enumerate(ranked_idx[:3]):
    print(f"  Rank {rank+1} (score={scores[idx]:.3f}):")
    print(f"    {pension_corpus[idx][:120]}...\n")

## 4. Dense Retrieval Recap

Dense retrieval uses a bi-encoder: both query and document are independently embedded into a vector space, then retrieved by cosine similarity (or inner product for normalised vectors).

We use **BAAI/bge-small-en-v1.5** — a strong 33M parameter model with a BGE query prefix.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

print("Loading BAAI/bge-small-en-v1.5...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Embed the pension corpus (documents — no prefix)
corpus_embeddings = embed_model.encode(
    pension_corpus,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
)
print(f"Corpus embedding shape: {corpus_embeddings.shape}")

In [ ]:
def dense_search(query: str, corpus_embs: np.ndarray, top_k: int = 5) -> list[int]:
    """Return indices of top-k documents by cosine similarity."""
    q_emb = embed_model.encode(
        [BGE_PREFIX + query], normalize_embeddings=True
    )[0]
    scores = corpus_embs @ q_emb
    return list(np.argsort(scores)[::-1][:top_k])

test_query = "Article 14(2) maximum recovery period"
dense_results = dense_search(test_query, corpus_embeddings, top_k=5)

print(f"Query: {test_query!r}\n")
print("Top-5 Dense results:")
for rank, idx in enumerate(dense_results):
    print(f"  Rank {rank+1}: [{idx}] {pension_corpus[idx][:100]}...")

## 5. Reciprocal Rank Fusion (RRF)

RRF merges two (or more) ranked lists without needing to normalise scores across different scales.

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k=60$ is a stabilising constant (avoids over-rewarding rank-1 documents). This was proposed by Cormack et al. (2009) and has become the standard fusion technique in production RAG systems.

In [ ]:
def rrf_merge(rankings: list[list[int]], k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)


def bm25_search(query: str, bm25_index: BM25Okapi, top_k: int = 10) -> list[int]:
    scores = bm25_index.get_scores(query.lower().split())
    return list(np.argsort(scores)[::-1][:top_k])


# Demonstrate RRF on our Article 14(2) query
k_retrieval = 10
bm25_ranking = bm25_search(test_query, bm25_pension, top_k=k_retrieval)
dense_ranking = dense_search(test_query, corpus_embeddings, top_k=k_retrieval)
hybrid_ranking = rrf_merge([bm25_ranking, dense_ranking], k=60)

print(f"Query: {test_query!r}\n")
print(f"{'Rank':<6} {'BM25 idx':<12} {'Dense idx':<12} {'Hybrid idx':<12}")
print("-" * 44)
for i in range(5):
    bm25_i  = bm25_ranking[i]  if i < len(bm25_ranking)  else "-"
    dense_i = dense_ranking[i] if i < len(dense_ranking) else "-"
    hyb_i   = hybrid_ranking[i] if i < len(hybrid_ranking) else "-"
    print(f"{i+1:<6} {str(bm25_i):<12} {str(dense_i):<12} {str(hyb_i):<12}")

## 6. Evaluation: BM25 vs Dense vs Hybrid

We measure **Recall@5**: for each question, does the ground-truth passage appear in the top-5 results?

In [ ]:
def recall_at_k(
    query: str,
    ground_truth_context: str,
    retrieved_indices: list[int],
    corpus: list[str],
    k: int = 5,
) -> bool:
    """Return True if ground_truth_context is retrieved in top-k results."""
    top_k_docs = [corpus[i] for i in retrieved_indices[:k] if i < len(corpus)]
    # Use substring match — ground truth may be a substring of a longer chunk
    for doc in top_k_docs:
        if ground_truth_context[:80] in doc or doc[:80] in ground_truth_context:
            return True
    return False


bm25_hits, dense_hits, hybrid_hits = 0, 0, 0
n_eval = 0

for item in golden_dataset:
    query = item["question"]
    for gt_context in item["contexts"]:
        if gt_context not in pension_corpus:
            continue  # skip if context not in our mini corpus
        n_eval += 1

        bm25_r  = bm25_search(query, bm25_pension, top_k=20)
        dense_r = dense_search(query, corpus_embeddings, top_k=20)
        hybrid_r = rrf_merge([bm25_r, dense_r], k=60)

        bm25_hits   += recall_at_k(query, gt_context, bm25_r,   pension_corpus)
        dense_hits  += recall_at_k(query, gt_context, dense_r,  pension_corpus)
        hybrid_hits += recall_at_k(query, gt_context, hybrid_r, pension_corpus)

print(f"Evaluation over {n_eval} query-context pairs\n")
print(f"{'Method':<12}  {'Recall@5':>10}")
print("-" * 26)
print(f"{'BM25':<12}  {bm25_hits/n_eval:>10.3f}")
print(f"{'Dense':<12}  {dense_hits/n_eval:>10.3f}")
print(f"{'Hybrid (RRF)':<12}  {hybrid_hits/n_eval:>10.3f}")

## 7. Using the `rag_pipeline.retriever` Module

The `HybridRetriever` class in `../rag_pipeline/retriever.py` integrates everything we've built above. It expects a Chroma collection and a BM25 index — both built by `Indexer`.

In [ ]:
import sys
sys.path.insert(0, "..")

from rag_pipeline.config import RAGConfig
from rag_pipeline.indexer import Indexer
from rag_pipeline.retriever import HybridRetriever

# Build a config with hybrid retrieval enabled
config = RAGConfig(
    chunking_strategy="recursive",
    chunk_size=400,
    chunk_overlap=40,
    initial_retrieval_k=10,
    final_top_k=5,
    use_bm25=True,
    use_reranking=False,  # disable reranking for now — covered in week2
    chroma_persist_dir="./chroma_week1",
)

indexer = Indexer(config)

# Build synthetic documents from the golden dataset contexts
docs = [{"page_content": ctx, "metadata": {"source": "regulation", "type": "text"}}
        for ctx in pension_corpus]

# Build both indexes
print("Building Chroma index...")
chroma_collection = indexer.build_index(docs)

print("Building BM25 index...")
bm25_index = indexer.build_bm25_index(docs)

chunks_text = [d["page_content"] for d in docs]

# Instantiate HybridRetriever
retriever = HybridRetriever(
    config=config,
    chroma_store=chroma_collection,
    bm25_index=bm25_index,
    chunks_text=chunks_text,
)

print("\nRetriever ready!")

In [ ]:
# Use the retriever on our Article 14(2) test
results = retriever.retrieve("Article 14(2) maximum recovery period ten years")

print("Top-5 results from HybridRetriever:\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] {doc['page_content'][:120]}...")
    print(f"       metadata: {doc['metadata']}\n")

## 8. Key Takeaways + Exercises

### Takeaways

| | BM25 | Dense | Hybrid (RRF) |
|---|---|---|---|
| Exact match ("Article 14(2)") | Strong | Weak | Strong |
| Semantic similarity | Weak | Strong | Strong |
| Paraphrase queries | Weak | Strong | Strong |
| Cross-lingual | No | Yes (multilingual model) | Partial |
| Speed | Very fast | Fast | Fast |
| Complexity | Low | Medium | Medium |

**RRF** is the recommended fusion strategy because:
- No score normalisation needed
- Robust to outlier scores
- Simple to implement and tune (only $k=60$)

### Exercises

1. **Tune RRF k**: Try `k` values of 10, 30, 60, 100. How does it affect Recall@5?
2. **Query types**: Test both exact-match queries (article numbers) and semantic queries ("what are the ESG requirements"). Which retriever wins each category?
3. **Weight BM25 vs dense**: Instead of equal weighting in RRF, try weighting BM25 results 2x for exact-match queries. Implement a simple query classifier (contains digits and article references → BM25-heavy).
4. **Extend the corpus**: Add more passages from the `golden_dataset.json` IPS section. Does Recall@5 change?